In [ ]:
import pyspark.sql.functions as F

print("=== Generación de Capa Gold (Normalización de Modelos) ===")

FACTOR_CAMBIO_PLN_A_EUR = 0.23  # Parámetro de conversión monetaria

# Cargar capas intermedias
df_de = spark.read.table("silver_precios_alemania")
df_es = spark.read.table("silver_precios_espana")
df_ro = spark.read.table("silver_precios_rumania")
df_pl = spark.read.table("silver_precios_polonia")

# Homogeneizar Polonia: Aplicar conversión de divisa y promediar a nivel horario (PT60M)
df_pl_convertido = df_pl.withColumn("precio_mwh", F.col("precio_mwh") * FACTOR_CAMBIO_PLN_A_EUR)

df_pl_horario = df_pl_convertido.withColumn("timestamp_utc", F.date_trunc("hour", F.col("timestamp_utc"))) \
                     .groupBy("timestamp_utc", "pais", "fuente") \
                     .agg(F.round(F.avg("precio_mwh"), 2).alias("precio_mwh")) \
                     .withColumn("timestamp_utc", F.col("timestamp_utc").cast("string"))

# Unión global del modelo unificado en Euros
df_gold_final = df_de.unionByName(df_es).unionByName(df_ro).unionByName(df_pl_horario)

# Guardar en la tabla maestra Delta de negocio
df_gold_final.write.format("delta").mode("overwrite").saveAsTable("gold_precios_horarios_europa")
print("💾 Tabla Gold Delta 'gold_precios_horarios_europa' persistida.")

In [ ]:
print("📤 Exportando snapshot analítico a la sección de archivos (Files)...")

df_export = spark.read.table("gold_precios_horarios_europa")

# Coalesce(1) unifica el particionado de Spark en un único archivo plano accesible
df_export.coalesce(1).write \
         .mode("overwrite") \
         .option("header", "true") \
         .csv("Files/gold_precios_horarios_europa_export")

print("✅ Proceso completado. Listo para descargar desde el explorador del Lakehouse.")